from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
# ─────────────── SETUP ───────────────
driver = webdriver.Chrome()
driver.get("https://www.euronics.it/")
time.sleep(3)

def close_cookie_popup(driver):
    try:
        close = driver.find_element(By.CSS_SELECTOR, "#onetrust-close-btn-container button")
        close.click()
        time.sleep(2)
    except:
        pass

close_cookie_popup(driver)

In [ ]:
# ─────────────── find subcategories ───────────────
soup = BeautifulSoup(driver.page_source, "html.parser")
subcategory_elements = soup.select("li.dropdown-item.dropdown a")

subcategories = []
for elem in subcategory_elements:
    name = elem.get_text(strip=True)
    link = elem.get("href")
    if link and not link.startswith("http"):
        link = "https://www.euronics.it" + link
    subcategories.append((name, link))

print(f" found {len(subcategories)} subcategories")


In [ ]:

# ─────────────── functions ───────────────
def scroll_down(driver):
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".product-tile"))
    )
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

def go_to_next_page(driver):
    try:
        current_url = driver.current_url
        next_button = driver.find_element(By.CSS_SELECTOR, 'button.page-link.metanext')
        if next_button.is_enabled():
            driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(3)
            return driver.current_url != current_url
    except:
        return False

def extract_items(driver, category_name):
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    items = []
    product_cards = soup.select(".product-tile")
    print(f" {category_name}: {len(product_cards)} products found")
    for card in product_cards:
        title = card.select_one("span.tile-name")
        price = card.select_one("span.price-formatted")
        items.append({
            "category": category_name,
            "product": title.get_text(strip=True) if title else "error product",
            "price": price.get_text(strip=True) if price else "error price"
        })
    return items

def scrape_category(driver, name, link):
    try:
        driver.get(link)
        time.sleep(2)
        close_cookie_popup(driver)
        all_items = []
        page = 1
        while True:
            print(f" {name} – page {page}")
            scroll_down(driver)
            prodotti = extract_items(driver, name)
            all_items.extend(prodotti)
            if not go_to_next_page(driver):
                break
            page += 1
        return all_items
    except Exception as e:
        print(f"Error in category {name}: {e}")
        return []

In [ ]:
# ───────────────  SCRAPING  ───────────────

all_data = []
for name, link in subcategories:
    print(f"{name}: {link}")
    prodotti = scrape_category(driver, name, link)
    all_data.extend(prodotti)

driver.quit()

# ─────────────── Save ───────────────

df = pd.DataFrame(all_data)
print(f" products collected: {len(df)}")
df.head(10)

#  CSV
df.to_csv("prodotti_euronics_completi.csv", index=False)


📚 Trovate 448 sottocategorie

🔗 Entrando in Computer Portatili → https://www.euronics.it/informatica/computer-portatili/
📄 Computer Portatili – Pagina 1
🔍 Computer Portatili: 48 prodotti trovati
📄 Computer Portatili – Pagina 2
🔍 Computer Portatili: 48 prodotti trovati
📄 Computer Portatili – Pagina 3
🔍 Computer Portatili: 48 prodotti trovati
📄 Computer Portatili – Pagina 4
🔍 Computer Portatili: 48 prodotti trovati
📄 Computer Portatili – Pagina 5
🔍 Computer Portatili: 48 prodotti trovati
📄 Computer Portatili – Pagina 6
🔍 Computer Portatili: 20 prodotti trovati

🔗 Entrando in Notebook → https://www.euronics.it/informatica/computer-portatili/notebook/
📄 Notebook – Pagina 1
🔍 Notebook: 48 prodotti trovati
📄 Notebook – Pagina 2
🔍 Notebook: 48 prodotti trovati
📄 Notebook – Pagina 3
🔍 Notebook: 48 prodotti trovati
📄 Notebook – Pagina 4
🔍 Notebook: 48 prodotti trovati
📄 Notebook – Pagina 5
🔍 Notebook: 48 prodotti trovati
📄 Notebook – Pagina 6
🔍 Notebook: 3 prodotti trovati

🔗 Entrando in Notebo

In [ ]:
df.head(5)

,categoria,nome,prezzo
0,Computer Portatili,HP - Notebook Gaming VICTUS 15-FA1057NL Intel ...,"€ 899,00"
1,Computer Portatili,SAMSUNG - Notebook GALAXY BOOK4 Core i7-Gray,"€ 699,00"
2,Computer Portatili,"LENOVO - LOQ Notebook 15,6"" Intel i7 32GB 1TB…...","€ 1.399,00"
3,Computer Portatili,ACER - Notebook A315-44P-R3CA-Silver,"€ 624,00"
4,Computer Portatili,"SAMSUNG - GALAXY CHROMEBOOK GO 14""-Silver","€ 269,00"
